# Dataset Pre-processing: Courier Earnings Analysis

In this notebook, I perform initial preparation of raw delivery reports:
- loading and concatenating weekly CSV files;
- removing summary rows (`ВСЬОГО`), missing values, and duplicates;
- converting numeric columns and timestamps to proper data types;
- creating a clean dataset for further EDA and visualization.

---

## 1. Importing Libraries and Environment Setup

In [197]:
import pandas as pd
import matplotlib.pyplot as pt
import seaborn as sns
import numpy as np
import os
import re

pd.set_option("display.max_rows", None)

---

## 2. Loading Raw CSV Files
Reading all files from the `data` folder into a list of DataFrames and inspecting the structure of the first weekly report.

In [199]:
folder_path = 'D:\\Programming\\Repository\\courier-earnings-analysis\\data'

list_name_data = os.listdir(folder_path)

df_list = []
q = 0

for i in list_name_data:
    if i.endswith('.csv'):
        df_list.append(pd.read_csv(f"D:\\Programming\\Repository\\courier-earnings-analysis\\data\\{i}"))
        df_list[q].drop(columns = ['На адресу'], inplace =True)
        q += 1
    
        

df_list[0]

,№,Дата,Час доставки,ID замовлення,Партнер,Рахунок клієнту (uah),Збір маркетплейсу (uah),Очікує компенсації (uah),ПДВ (uah),Всього (uah),Готівкою партнеру (uah),Готівка від клієнта (uah)
0,1.0,2026-07-27,20:01,J7HUK,KFC Sky Park,113.80,26.4728,-,-,140.27,-,-
1,2.0,2026-07-27,21:02,XNMUK,La cucina вул. Янгеля,104.20,63.8455,-,-,168.05,-,-
2,3.0,2026-07-27,22:46,H69KZ,SAMANDAR DONER,107.10,104.8525,-,-,211.95,-,-
3,NaN,2026-07-27,NaN,NaN,NaN,325.10,195.17,-,-,520.27,-,-
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,2026-07-28,20:12,4N4JQ,Шаурмішка 600-річчя,98.20,-1.8678,-,-,96.33,-,-
6,5.0,2026-07-28,20:53,QCKPC,KFC Мегамолл,97.40,33.4118,-,-,130.81,-,-
7,6.0,2026-07-28,21:46,7KEBL,Тареля,121.40,69.9979,-,-,191.40,-,-
8,7.0,2026-07-28,22:29,V4Y7F,Sushi Master Соборна,109.20,41.8158,-,-,151.02,-,-
9,8.0,2026-07-28,23:01,XQGKI,OKKO CAFE вул. Костянтина Василенка,53.60,79.6162,-,-,133.22,-,-


## 3. Removing Summary Rows and Concatenating Files
The raw reports contain blank rows and summary rows (`ВСЬОГО`, `УСЬОГО ЗА ТИЖДЕНЬ`) where `ID замовлення` is missing. Removing them before merging the tables.

In [200]:
for i in range(0, len(df_list)):
    df_list[i].dropna(subset = ['ID замовлення'], inplace = True)
    print (f'Чи очищений {i+1} файл від сміттєвих даних: {"True" if len(pd.unique(df_list[i]['ID замовлення'].notna())) == 1 else "False"}')   

Чи очищений 1 файл від сміттєвих даних: True
Чи очищений 2 файл від сміттєвих даних: True
Чи очищений 3 файл від сміттєвих даних: True
Чи очищений 4 файл від сміттєвих даних: True
Чи очищений 5 файл від сміттєвих даних: True
Чи очищений 6 файл від сміттєвих даних: True
Чи очищений 7 файл від сміттєвих даних: True


In [204]:
df = pd.concat(df_list, ignore_index = True)
df.head(10)

,№,Дата,Час доставки,ID замовлення,Партнер,Рахунок клієнту (uah),Збір маркетплейсу (uah),Очікує компенсації (uah),ПДВ (uah),Всього (uah),Готівкою партнеру (uah),Готівка від клієнта (uah)
0,1.0,2026-07-27,20:01,J7HUK,KFC Sky Park,113.80,26.4728,-,-,140.27,-,-
1,2.0,2026-07-27,21:02,XNMUK,La cucina вул. Янгеля,104.20,63.8455,-,-,168.05,-,-
2,3.0,2026-07-27,22:46,H69KZ,SAMANDAR DONER,107.10,104.8525,-,-,211.95,-,-
3,4.0,2026-07-28,20:12,4N4JQ,Шаурмішка 600-річчя,98.20,-1.8678,-,-,96.33,-,-
4,5.0,2026-07-28,20:53,QCKPC,KFC Мегамолл,97.40,33.4118,-,-,130.81,-,-
5,6.0,2026-07-28,21:46,7KEBL,Тареля,121.40,69.9979,-,-,191.40,-,-
6,7.0,2026-07-28,22:29,V4Y7F,Sushi Master Соборна,109.20,41.8158,-,-,151.02,-,-
7,8.0,2026-07-28,23:01,XQGKI,OKKO CAFE вул. Костянтина Василенка,53.60,79.6162,-,-,133.22,-,-
8,9.0,2026-07-29,20:45,OWWDV,KFC Мегамолл,69.10,14.4590,-,-,83.56,-,-
9,10.0,2026-07-29,21:48,TGK5R,KFC Мегамолл,36.90,51.1286,-,-,88.03,-,-


In [205]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   №                          146 non-null    float64
 1   Дата                       146 non-null    object 
 2   Час доставки               146 non-null    object 
 3   ID замовлення              146 non-null    object 
 4   Партнер                    146 non-null    object 
 5   Рахунок клієнту (uah)      146 non-null    object 
 6   Збір маркетплейсу (uah)    146 non-null    object 
 7   Очікує компенсації (uah)   146 non-null    object 
 8   ПДВ (uah)                  146 non-null    object 
 9   Всього (uah)               146 non-null    object 
 10  Готівкою партнеру (uah)    146 non-null    object 
 11  Готівка від клієнта (uah)  146 non-null    object 
dtypes: float64(1), object(11)
memory usage: 13.8+ KB


## 4. Handling Numeric Columns and Missing Values
Replacing dashes (`-`) with `NaN`, removing extra spaces, standardizing the decimal separator (`,` → `.`), and converting financial columns to `float`.

In [206]:
def to_nan(opt):
    return np.nan if opt == '-' else opt

index_column_money = pd.array(['Рахунок клієнту (uah)', 'Збір маркетплейсу (uah)', 'Всього (uah)', 'Очікує компенсації (uah)', 'ПДВ (uah)', 'Готівкою партнеру (uah)', 'Готівка від клієнта (uah)'])

for i in index_column_money:
    df[i] = df[i].str.replace(',', '.')
    df[i] = df[i].str.replace(' ', '')
    df[i] = df[i].apply(to_nan)
    df[i] = pd.to_numeric(df[i], errors="coerce")

df['ID замовлення'] = df['ID замовлення'].str.replace(' ', '')

In [207]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   №                          146 non-null    float64
 1   Дата                       146 non-null    object 
 2   Час доставки               146 non-null    object 
 3   ID замовлення              146 non-null    object 
 4   Партнер                    146 non-null    object 
 5   Рахунок клієнту (uah)      146 non-null    float64
 6   Збір маркетплейсу (uah)    142 non-null    float64
 7   Очікує компенсації (uah)   0 non-null      float64
 8   ПДВ (uah)                  0 non-null      float64
 9   Всього (uah)               146 non-null    float64
 10  Готівкою партнеру (uah)    0 non-null      float64
 11  Готівка від клієнта (uah)  10 non-null     float64
dtypes: float64(8), object(4)
memory usage: 13.8+ KB


## 5. Finding and Removing Duplicates
Checking for duplicates by unique `ID замовлення`, masking duplicate rows, and dropping empty rows.

In [208]:
df[df['ID замовлення'].duplicated()]

,№,Дата,Час доставки,ID замовлення,Партнер,Рахунок клієнту (uah),Збір маркетплейсу (uah),Очікує компенсації (uah),ПДВ (uah),Всього (uah),Готівкою партнеру (uah),Готівка від клієнта (uah)
65,1.0,2026-08-08,18:09,KZGMN,"VARUS, пр. Коцюбинського, 70",25.00,NaN,NaN,NaN,25.00,NaN,NaN
66,2.0,2026-08-08,19:58,SZ67O,KFC Мегамолл,25.00,NaN,NaN,NaN,25.00,NaN,NaN
87,1.0,2026-08-16,19:54,VWTQE,McDonald's Соборна,25.00,NaN,NaN,NaN,25.00,NaN,NaN
136,1.0,2026-09-20,20:33,GW2GL,"VARUS, пр. Коцюбинського, 70",14.74,NaN,NaN,NaN,14.74,NaN,NaN


In [209]:
df.mask(df['ID замовлення'].duplicated(), inplace = True)
df[df['ID замовлення'].isna()]

,№,Дата,Час доставки,ID замовлення,Партнер,Рахунок клієнту (uah),Збір маркетплейсу (uah),Очікує компенсації (uah),ПДВ (uah),Всього (uah),Готівкою партнеру (uah),Готівка від клієнта (uah)
65,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
87,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
136,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [210]:
df.dropna(axis = 'index', how = 'all', inplace = True)

In [211]:
df.head(10)

,№,Дата,Час доставки,ID замовлення,Партнер,Рахунок клієнту (uah),Збір маркетплейсу (uah),Очікує компенсації (uah),ПДВ (uah),Всього (uah),Готівкою партнеру (uah),Готівка від клієнта (uah)
0,1.0,2026-07-27,20:01,J7HUK,KFC Sky Park,113.8,26.4728,NaN,NaN,140.27,NaN,NaN
1,2.0,2026-07-27,21:02,XNMUK,La cucina вул. Янгеля,104.2,63.8455,NaN,NaN,168.05,NaN,NaN
2,3.0,2026-07-27,22:46,H69KZ,SAMANDAR DONER,107.1,104.8525,NaN,NaN,211.95,NaN,NaN
3,4.0,2026-07-28,20:12,4N4JQ,Шаурмішка 600-річчя,98.2,-1.8678,NaN,NaN,96.33,NaN,NaN
4,5.0,2026-07-28,20:53,QCKPC,KFC Мегамолл,97.4,33.4118,NaN,NaN,130.81,NaN,NaN
5,6.0,2026-07-28,21:46,7KEBL,Тареля,121.4,69.9979,NaN,NaN,191.40,NaN,NaN
6,7.0,2026-07-28,22:29,V4Y7F,Sushi Master Соборна,109.2,41.8158,NaN,NaN,151.02,NaN,NaN
7,8.0,2026-07-28,23:01,XQGKI,OKKO CAFE вул. Костянтина Василенка,53.6,79.6162,NaN,NaN,133.22,NaN,NaN
8,9.0,2026-07-29,20:45,OWWDV,KFC Мегамолл,69.1,14.4590,NaN,NaN,83.56,NaN,NaN
9,10.0,2026-07-29,21:48,TGK5R,KFC Мегамолл,36.9,51.1286,NaN,NaN,88.03,NaN,NaN


In [212]:
df[df['ID замовлення'].duplicated()]

,№,Дата,Час доставки,ID замовлення,Партнер,Рахунок клієнту (uah),Збір маркетплейсу (uah),Очікує компенсації (uah),ПДВ (uah),Всього (uah),Готівкою партнеру (uah),Готівка від клієнта (uah)


## 6. Creating Delivery Timestamp and Trimming Columns
Combining delivery date and time into a single `Повна дата` (`datetime64`) column and dropping unneeded, empty, or overly messy columns.

In [213]:
df['Повна дата'] = pd.to_datetime(df['Дата'] + ' ' + df['Час доставки'], format = '%Y-%m-%d %H:%M')

In [214]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 142 entries, 0 to 145
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   №                          142 non-null    float64       
 1   Дата                       142 non-null    object        
 2   Час доставки               142 non-null    object        
 3   ID замовлення              142 non-null    object        
 4   Партнер                    142 non-null    object        
 5   Рахунок клієнту (uah)      142 non-null    float64       
 6   Збір маркетплейсу (uah)    142 non-null    float64       
 7   Очікує компенсації (uah)   0 non-null      float64       
 8   ПДВ (uah)                  0 non-null      float64       
 9   Всього (uah)               142 non-null    float64       
 10  Готівкою партнеру (uah)    0 non-null      float64       
 11  Готівка від клієнта (uah)  10 non-null     float64       
 12  Повна дата   

In [215]:
df = df.drop(columns = ['Дата','Час доставки','Очікує компенсації (uah)','ПДВ (uah)','Готівкою партнеру (uah)', '№'])
df.head(10)

,ID замовлення,Партнер,Рахунок клієнту (uah),Збір маркетплейсу (uah),Всього (uah),Готівка від клієнта (uah),Повна дата
0,J7HUK,KFC Sky Park,113.8,26.4728,140.27,NaN,2026-07-27 20:01:00
1,XNMUK,La cucina вул. Янгеля,104.2,63.8455,168.05,NaN,2026-07-27 21:02:00
2,H69KZ,SAMANDAR DONER,107.1,104.8525,211.95,NaN,2026-07-27 22:46:00
3,4N4JQ,Шаурмішка 600-річчя,98.2,-1.8678,96.33,NaN,2026-07-28 20:12:00
4,QCKPC,KFC Мегамолл,97.4,33.4118,130.81,NaN,2026-07-28 20:53:00
5,7KEBL,Тареля,121.4,69.9979,191.40,NaN,2026-07-28 21:46:00
6,V4Y7F,Sushi Master Соборна,109.2,41.8158,151.02,NaN,2026-07-28 22:29:00
7,XQGKI,OKKO CAFE вул. Костянтина Василенка,53.6,79.6162,133.22,NaN,2026-07-28 23:01:00
8,OWWDV,KFC Мегамолл,69.1,14.4590,83.56,NaN,2026-07-29 20:45:00
9,TGK5R,KFC Мегамолл,36.9,51.1286,88.03,NaN,2026-07-29 21:48:00


In [216]:
df[df['Готівка від клієнта (uah)'].notna()]

,ID замовлення,Партнер,Рахунок клієнту (uah),Збір маркетплейсу (uah),Всього (uah),Готівка від клієнта (uah),Повна дата
112,EP6JG,Валенсія,49.5,72.3070,121.81,322.10,2026-09-17 16:33:00
118,3JGXZ,ПАБ Гаштет ввул. Зодчих,145.2,76.2760,221.48,886.20,2026-09-17 22:07:00
119,UHTVV,Smak Svitu,76.4,25.1036,101.50,986.40,2026-09-18 14:33:00
120,QTQJZ,Yoshi вул. Соборна,58.3,58.4968,116.80,298.30,2026-09-18 15:36:00
122,K9K4E,Шаурмішка Пирогова,37.1,38.9074,76.01,233.45,2026-09-18 17:13:00
123,RW41E,Miami pizza,93.3,-6.5028,86.80,762.20,2026-09-18 18:11:00
126,RBTSU,Світ сендвіч& гриль,49.2,35.9738,85.17,404.20,2026-09-18 20:16:00
131,NRZ3A,O'Cake пр. Коцюбинського 2-A,135.6,26.4720,162.07,827.70,2026-09-20 17:36:00
133,A7DDE,КОСМОС,129.7,8.7672,138.47,918.70,2026-09-20 20:01:00
139,S757V,Chili pizza Коцюбинського,111.6,24.4728,136.07,746.10,2026-09-25 21:04:00


In [217]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 142 entries, 0 to 145
Data columns (total 7 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   ID замовлення              142 non-null    object        
 1   Партнер                    142 non-null    object        
 2   Рахунок клієнту (uah)      142 non-null    float64       
 3   Збір маркетплейсу (uah)    142 non-null    float64       
 4   Всього (uah)               142 non-null    float64       
 5   Готівка від клієнта (uah)  10 non-null     float64       
 6   Повна дата                 142 non-null    datetime64[ns]
dtypes: datetime64[ns](1), float64(4), object(2)
memory usage: 8.9+ KB


## 7. Rounding Monetary Values and Final Dataset Check
Rounding amounts to two decimal places and inspecting the resulting table before export.

In [218]:
name_of_columns = ['Рахунок клієнту (uah)','Збір маркетплейсу (uah)','Всього (uah)','Готівка від клієнта (uah)']

for i in name_of_columns:
    df[i] = df[i].round(2)

In [219]:
df.head(10)

,ID замовлення,Партнер,Рахунок клієнту (uah),Збір маркетплейсу (uah),Всього (uah),Готівка від клієнта (uah),Повна дата
0,J7HUK,KFC Sky Park,113.8,26.47,140.27,NaN,2026-07-27 20:01:00
1,XNMUK,La cucina вул. Янгеля,104.2,63.85,168.05,NaN,2026-07-27 21:02:00
2,H69KZ,SAMANDAR DONER,107.1,104.85,211.95,NaN,2026-07-27 22:46:00
3,4N4JQ,Шаурмішка 600-річчя,98.2,-1.87,96.33,NaN,2026-07-28 20:12:00
4,QCKPC,KFC Мегамолл,97.4,33.41,130.81,NaN,2026-07-28 20:53:00
5,7KEBL,Тареля,121.4,70.00,191.40,NaN,2026-07-28 21:46:00
6,V4Y7F,Sushi Master Соборна,109.2,41.82,151.02,NaN,2026-07-28 22:29:00
7,XQGKI,OKKO CAFE вул. Костянтина Василенка,53.6,79.62,133.22,NaN,2026-07-28 23:01:00
8,OWWDV,KFC Мегамолл,69.1,14.46,83.56,NaN,2026-07-29 20:45:00
9,TGK5R,KFC Мегамолл,36.9,51.13,88.03,NaN,2026-07-29 21:48:00


In [220]:
df.reset_index(drop=True, inplace=True)

## 8. Saving the Cleaned Dataset
Exporting the final table to CSV for exploratory data analysis (EDA) and dashboarding.

In [221]:
df.to_csv('../data/cleaned_deliveries.csv', index=False, encoding="utf-8-sig")